# SparkCity Weather Feature — Day 2

**Owner:** Matthew Baise  
**Focus:** Data quality, cleaning, missing-data analysis, outlier detection, and standardization

## Purpose

This notebook builds a reproducible PySpark cleaning pipeline for the SparkCity weather dataset. The original source data remains unchanged.

In [67]:
from pathlib import Path

from pyspark.sql import DataFrame, SparkSession, Window
from pyspark.sql import functions as F

from sparkcityx.data_quality import validate_dataframe
from sparkcityx.loaders import load_dataset

In [68]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("SparkCity-Weather-Day2")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Spark master: {spark.sparkContext.master}")

Spark version: 4.2.0
Spark master: local[*]


In [69]:
weather_path = Path("../data/raw/weather_data.parquet")

if not weather_path.exists():
    weather_path = Path("data/raw/weather_data.parquet")

weather_df = load_dataset(spark, weather_path).cache()
record_count = weather_df.count()

print(f"Source: {weather_path.resolve()}")
print(f"Records loaded: {record_count:,}")

Source: /Users/matthew/Projects/SparkCity_Capstone/data/raw/weather_data.parquet
Records loaded: 36,000


26/09/14 16:31:27 WARN CacheManager: Asked to cache already cached data.


In [70]:
baseline_report = validate_dataframe(weather_df, "weather")

print(f"Baseline valid: {baseline_report['valid']}")
print(f"Null counts: {baseline_report['null_counts']}")
print(f"Duplicate count: {baseline_report['duplicate_count']}")
print(f"Range violations: {baseline_report['range_violations']}")

Baseline valid: True
Null counts: {}
Duplicate count: 0
Range violations: {}


## 1. Missing Values and Duplicate Readings

Weather records are uniquely identified by `station_id` and `timestamp`. This section measures missing values and detects repeated keys before cleaning.

In [71]:
def profile_missing_values(df: DataFrame) -> DataFrame:
    total_rows = df.count()

    expressions = []

    for column in df.columns:
        null_count = F.sum(
            F.when(F.col(column).isNull(), 1).otherwise(0)
        ).alias(f"{column}_nulls")

        expressions.append(null_count)

    result = df.agg(*expressions)

    for column in df.columns:
        result = result.withColumn(
            f"{column}_missing_percent",
            F.round(
                F.col(f"{column}_nulls") / F.lit(total_rows) * 100,
                2,
            ),
        )

    return result

In [72]:
missing_profile_df = profile_missing_values(weather_df)
missing_profile_df.show(truncate=False)

+----------------+---------------+------------------+------------------+-----------------+--------------+----------------+--------------------+-------------------+--------------+--------------------------+-------------------------+----------------------------+----------------------------+---------------------------+------------------------+--------------------------+------------------------------+-----------------------------+------------------------+
|station_id_nulls|timestamp_nulls|location_lat_nulls|location_lon_nulls|temperature_nulls|humidity_nulls|wind_speed_nulls|wind_direction_nulls|precipitation_nulls|pressure_nulls|station_id_missing_percent|timestamp_missing_percent|location_lat_missing_percent|location_lon_missing_percent|temperature_missing_percent|humidity_missing_percent|wind_speed_missing_percent|wind_direction_missing_percent|precipitation_missing_percent|pressure_missing_percent|
+----------------+---------------+------------------+------------------+----------------

In [73]:
def find_weather_duplicates(df: DataFrame) -> DataFrame:
    return (
        df
        .groupBy("station_id", "timestamp")
        .count()
        .filter(F.col("count") > 1)
        .orderBy("station_id", "timestamp")
    )


duplicate_df = find_weather_duplicates(weather_df)

print(f"Duplicate keys: {duplicate_df.count():,}")
duplicate_df.show(10, truncate=False)

Duplicate keys: 0
+----------+---------+-----+
|station_id|timestamp|count|
+----------+---------+-----+
+----------+---------+-----+



## 2. Station Timestamp-Gap Analysis

A lag window compares each reading with the previous reading from the same station. This identifies the observed reporting interval and unusually long gaps.

In [74]:
station_window = Window.partitionBy("station_id").orderBy("timestamp")

weather_gap_df = (
    weather_df
    .withColumn(
        "previous_timestamp",
        F.lag("timestamp").over(station_window),
    )
    .withColumn(
        "gap_hours",
        (
            F.unix_timestamp("timestamp")
            - F.unix_timestamp("previous_timestamp")
        ) / 3600,
    )
)

weather_gap_df.select("gap_hours").summary(
    "count",
    "mean",
    "stddev",
    "min",
    "50%",
    "max",
).show(truncate=False)

+-------+---------+
|summary|gap_hours|
+-------+---------+
|count  |34800    |
|mean   |600.0    |
|stddev |0.0      |
|min    |600.0    |
|50%    |600.0    |
|max    |600.0    |
+-------+---------+



In [75]:
(
    weather_gap_df
    .filter(F.col("gap_hours").isNotNull())
    .groupBy("gap_hours")
    .count()
    .orderBy(F.desc("count"))
    .show(10, truncate=False)
)

+---------+-----+
|gap_hours|count|
+---------+-----+
|600.0    |34800|
+---------+-----+



In [76]:
interval_counts_df = (
    weather_gap_df
    .filter(F.col("gap_hours").isNotNull())
    .groupBy("gap_hours")
    .count()
    .orderBy(F.desc("count"), F.asc("gap_hours"))
)

observed_interval_hours = interval_counts_df.first()["gap_hours"]
outage_threshold_hours = observed_interval_hours * 1.5

print(f"Observed station interval: {observed_interval_hours:.2f} hours")
print(f"Outage threshold: {outage_threshold_hours:.2f} hours")

Observed station interval: 600.00 hours
Outage threshold: 900.00 hours


In [77]:
weather_gap_status_df = weather_gap_df.withColumn(
    "gap_status",
    F.when(
        F.col("previous_timestamp").isNull(),
        F.lit("FIRST_READING"),
    )
    .when(
        F.col("gap_hours") > outage_threshold_hours,
        F.lit("POSSIBLE_OUTAGE"),
    )
    .otherwise(F.lit("EXPECTED_INTERVAL")),
)

In [78]:
station_health_df = (
    weather_gap_status_df
    .groupBy("station_id")
    .agg(
        F.count("*").alias("reading_count"),
        F.min("timestamp").alias("first_reading"),
        F.max("timestamp").alias("last_reading"),
        F.max("gap_hours").alias("maximum_gap_hours"),
        F.sum(
            F.when(
                F.col("gap_status") == "POSSIBLE_OUTAGE",
                1,
            ).otherwise(0)
        ).alias("possible_outages"),
    )
    .orderBy(F.desc("possible_outages"), "station_id")
)

station_health_df.show(10, truncate=False)

+----------+-------------+-------------------+-------------------+-----------------+----------------+
|station_id|reading_count|first_reading      |last_reading       |maximum_gap_hours|possible_outages|
+----------+-------------+-------------------+-------------------+-----------------+----------------+
|WTH-0001  |30           |2025-01-01 00:00:00|2026-12-27 00:00:00|600.0            |0               |
|WTH-0002  |30           |2025-01-01 00:30:00|2026-12-27 00:30:00|600.0            |0               |
|WTH-0003  |30           |2025-01-01 01:00:00|2026-12-27 01:00:00|600.0            |0               |
|WTH-0004  |30           |2025-01-01 01:30:00|2026-12-27 01:30:00|600.0            |0               |
|WTH-0005  |30           |2025-01-01 02:00:00|2026-12-27 02:00:00|600.0            |0               |
|WTH-0006  |30           |2025-01-01 02:30:00|2026-12-27 02:30:00|600.0            |0               |
|WTH-0007  |30           |2025-01-01 03:00:00|2026-12-27 03:00:00|600.0           

In [79]:
station_health_df.agg(
    F.count("*").alias("stations_analyzed"),
    F.sum("possible_outages").alias("possible_outages"),
    F.min("reading_count").alias("minimum_readings"),
    F.max("reading_count").alias("maximum_readings"),
    F.avg("reading_count").alias("average_readings"),
).show(truncate=False)

+-----------------+----------------+----------------+----------------+----------------+
|stations_analyzed|possible_outages|minimum_readings|maximum_readings|average_readings|
+-----------------+----------------+----------------+----------------+----------------+
|1200             |0               |30              |30              |30.0            |
+-----------------+----------------+----------------+----------------+----------------+



## 3. Weather Domain Validation

Domain rules identify physically or structurally invalid readings. Temperature is excluded until its unit is formally confirmed.

In [80]:
def add_weather_domain_flags(df: DataFrame) -> DataFrame:
    return (
        df
        .withColumn(
            "invalid_location",
            ~F.col("location_lat").between(-90, 90)
            | ~F.col("location_lon").between(-180, 180),
        )
        .withColumn(
            "invalid_humidity",
            ~F.col("humidity").between(0, 100),
        )
        .withColumn(
            "invalid_wind_speed",
            F.col("wind_speed") < 0,
        )
        .withColumn(
            "invalid_wind_direction",
            ~F.col("wind_direction").between(0, 360),
        )
        .withColumn(
            "invalid_precipitation",
            F.col("precipitation") < 0,
        )
        .withColumn(
            "invalid_pressure",
            F.col("pressure") <= 0,
        )
    )


weather_domain_df = add_weather_domain_flags(weather_df)

In [81]:
domain_flag_columns = [
    "invalid_location",
    "invalid_humidity",
    "invalid_wind_speed",
    "invalid_wind_direction",
    "invalid_precipitation",
    "invalid_pressure",
]

weather_domain_df.agg(
    *[
        F.sum(
            F.when(F.col(column), 1).otherwise(0)
        ).alias(column)
        for column in domain_flag_columns
    ]
).show(truncate=False)

+----------------+----------------+------------------+----------------------+---------------------+----------------+
|invalid_location|invalid_humidity|invalid_wind_speed|invalid_wind_direction|invalid_precipitation|invalid_pressure|
+----------------+----------------+------------------+----------------------+---------------------+----------------+
|0               |0               |0                 |0                     |0                    |0               |
+----------------+----------------+------------------+----------------------+---------------------+----------------+



## 4. IQR Outlier Detection

IQR flags statistically unusual measurements. A statistical outlier is not automatically invalid and should not be deleted without investigation.

In [82]:
iqr_columns = [
    "temperature",
    "humidity",
    "wind_speed",
    "precipitation",
    "pressure",
]


def add_iqr_outlier_flags(
    df: DataFrame,
    columns: list[str],
) -> tuple[DataFrame, dict[str, dict[str, float]]]:
    result_df = df
    bounds = {}

    for column in columns:
        q1, q3 = df.approxQuantile(column, [0.25, 0.75], 0.01)
        iqr = q3 - q1
        lower_bound = q1 - (1.5 * iqr)
        upper_bound = q3 + (1.5 * iqr)

        bounds[column] = {
            "q1": q1,
            "q3": q3,
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
        }

        result_df = result_df.withColumn(
            f"{column}_iqr_outlier",
            ~F.col(column).between(lower_bound, upper_bound),
        )

    return result_df, bounds

In [83]:
weather_outlier_df, iqr_bounds = add_iqr_outlier_flags(
    weather_domain_df,
    iqr_columns,
)

for column, values in iqr_bounds.items():
    print(
        f"{column}: "
        f"lower={values['lower_bound']:.3f}, "
        f"upper={values['upper_bound']:.3f}"
    )

temperature: lower=7.815, upper=95.935
humidity: lower=26.620, upper=88.940
wind_speed: lower=-2.030, upper=29.490
precipitation: lower=-0.295, upper=0.492
pressure: lower=992.460, upper=1031.420


In [84]:
iqr_flag_columns = [
    f"{column}_iqr_outlier"
    for column in iqr_columns
]

weather_outlier_df.agg(
    *[
        F.sum(
            F.when(F.col(column), 1).otherwise(0)
        ).alias(column)
        for column in iqr_flag_columns
    ]
).show(truncate=False)

+-----------------------+--------------------+----------------------+-------------------------+--------------------+
|temperature_iqr_outlier|humidity_iqr_outlier|wind_speed_iqr_outlier|precipitation_iqr_outlier|pressure_iqr_outlier|
+-----------------------+--------------------+----------------------+-------------------------+--------------------+
|0                      |0                   |0                     |365                      |0                   |
+-----------------------+--------------------+----------------------+-------------------------+--------------------+



## 5. Cleaning and Imputation Pipeline

Only isolated missing measurements with both a previous and next station reading are imputed. Statistical outliers are retained because unusual weather can still be legitimate.

In [85]:
imputation_columns = [
    "temperature",
    "humidity",
    "wind_speed",
    "precipitation",
    "pressure",
]


def impute_isolated_weather_values(df: DataFrame) -> DataFrame:
    result_df = df
    station_window = Window.partitionBy("station_id").orderBy("timestamp")

    for column in imputation_columns:
        previous_column = f"_previous_{column}"
        next_column = f"_next_{column}"
        imputed_flag = f"{column}_was_imputed"

        result_df = (
            result_df
            .withColumn(
                previous_column,
                F.lag(column).over(station_window),
            )
            .withColumn(
                next_column,
                F.lead(column).over(station_window),
            )
            .withColumn(
                imputed_flag,
                F.col(column).isNull()
                & F.col(previous_column).isNotNull()
                & F.col(next_column).isNotNull(),
            )
            .withColumn(
                column,
                F.when(
                    F.col(imputed_flag),
                    (
                        F.col(previous_column)
                        + F.col(next_column)
                    ) / 2,
                ).otherwise(F.col(column)),
            )
            .drop(previous_column, next_column)
        )

    return result_df

In [86]:
weather_deduplicated_df = weather_df.dropDuplicates(
    ["station_id", "timestamp"]
)

weather_imputed_df = impute_isolated_weather_values(
    weather_deduplicated_df
)

weather_cleaning_df = add_weather_domain_flags(
    weather_imputed_df
)

for column, bounds in iqr_bounds.items():
    weather_cleaning_df = weather_cleaning_df.withColumn(
        f"{column}_iqr_outlier",
        ~F.col(column).between(
            bounds["lower_bound"],
            bounds["upper_bound"],
        ),
    )



In [87]:

invalid_domain_condition = F.lit(False)
for column in domain_flag_columns:
    invalid_domain_condition = (
        invalid_domain_condition | F.col(column)
    )

incomplete_condition = F.lit(False)
for column in weather_df.columns:
    incomplete_condition = (
        incomplete_condition | F.col(column).isNull()
    )

imputed_condition = F.lit(False)
for column in imputation_columns:
    imputed_condition = (
        imputed_condition
        | F.col(f"{column}_was_imputed")
    )

weather_classified_df = (
    weather_cleaning_df
    .withColumn(
        "cleaning_status",
        F.when(
            invalid_domain_condition,
            F.lit("INVALID_DOMAIN"),
        )
        .when(
            incomplete_condition,
            F.lit("INCOMPLETE"),
        )
        .when(
            imputed_condition,
            F.lit("IMPUTED"),
        )
        .otherwise(F.lit("CLEAN")),
    )
    .withColumn(
        "source_file",
        F.lit(weather_path.name),
    )
    .withColumn(
        "processed_at",
        F.current_timestamp(),
    )
)

In [88]:
clean_weather_df = weather_classified_df.filter(
    F.col("cleaning_status").isin("CLEAN", "IMPUTED")
)

quarantined_weather_df = weather_classified_df.filter(
    F.col("cleaning_status").isin(
        "INVALID_DOMAIN",
        "INCOMPLETE",
    )
)

weather_classified_df.groupBy(
    "cleaning_status"
).count().orderBy("cleaning_status").show()

print(f"Source rows: {weather_df.count():,}")
print(f"Deduplicated rows: {weather_deduplicated_df.count():,}")
print(f"Clean output rows: {clean_weather_df.count():,}")
print(f"Quarantined rows: {quarantined_weather_df.count():,}")

+---------------+-----+
|cleaning_status|count|
+---------------+-----+
|          CLEAN|36000|
+---------------+-----+

Source rows: 36,000
Deduplicated rows: 36,000
Clean output rows: 36,000
Quarantined rows: 0


## 6. Cleaned Dataset Validation and Findings

Invalid and incomplete records are quarantined rather than silently discarded. IQR outliers remain in the clean dataset with flags because unusual weather may be legitimate.

Measurement units were not converted because the source contract does not identify them. Unit confirmation is required before temperature, wind-speed, precipitation, or pressure conversions are appropriate.

In [89]:
clean_report = validate_dataframe(
    clean_weather_df.select(*weather_df.columns),
    "weather",
)

print(f"Clean dataset valid: {clean_report['valid']}")
print(f"Clean records: {clean_report['record_count']:,}")
print(f"Null counts: {clean_report['null_counts']}")
print(f"Duplicate count: {clean_report['duplicate_count']}")
print(f"Range violations: {clean_report['range_violations']}")

Clean dataset valid: True
Clean records: 36,000
Null counts: {}
Duplicate count: 0
Range violations: {}


In [90]:
clean_weather_df.agg(
    *[
        F.sum(
            F.when(F.col(column), 1).otherwise(0)
        ).alias(column)
        for column in iqr_flag_columns
    ]
).show(truncate=False)

+-----------------------+--------------------+----------------------+-------------------------+--------------------+
|temperature_iqr_outlier|humidity_iqr_outlier|wind_speed_iqr_outlier|precipitation_iqr_outlier|pressure_iqr_outlier|
+-----------------------+--------------------+----------------------+-------------------------+--------------------+
|0                      |0                   |0                     |365                      |0                   |
+-----------------------+--------------------+----------------------+-------------------------+--------------------+



In [91]:
print("DAY 2 WEATHER CLEANING SUMMARY")
print("=" * 50)
print(f"Source records: {weather_df.count():,}")
print(f"Duplicate records removed: {weather_df.count() - weather_deduplicated_df.count():,}")
print(f"Clean records: {clean_weather_df.count():,}")
print(f"Quarantined records: {quarantined_weather_df.count():,}")
print(f"Clean dataset valid: {clean_report['valid']}")
print(f"Observed station interval: {observed_interval_hours:.2f} hours")
print(f"Possible station outages: {station_health_df.agg(F.sum('possible_outages')).first()[0]:,}")

DAY 2 WEATHER CLEANING SUMMARY
Source records: 36,000
Duplicate records removed: 0
Clean records: 36,000
Quarantined records: 0
Clean dataset valid: True
Observed station interval: 600.00 hours
Possible station outages: 0


In [92]:
weather_df.unpersist()
spark.stop()
print("Spark session stopped successfully.")

Spark session stopped successfully.
